Task 1

In [ ]:
import pandas as pd

# Load the loans data with date columns parsed as datetime
loans_raw = pd.read_csv('data/library_loans_clean.csv', parse_dates=['loan_date', 'return_date'])
loans = loans_raw.copy()

# Check shape and column types
print("Shape:", loans.shape)
print("\nData types:")
print(loans.dtypes)

print("\nMissing values:")
print(loans.isna().sum())
print("\nMissing values (%):")
print((loans[['loan_date', 'return_date']].isna().mean() * 100).round(1))

# loan_date has 3 missing values (0.5%) and return_date has 36 missing values (6.0%)

Task 2

In [ ]:
print(loans[loans['loan_date'].isna()])

# Count missing return_date depending on whether the book is returned or not
missing_and_returned = (loans['return_date'].isna() & (loans['returned'] == True)).sum()
missing_and_not_returned = (loans['return_date'].isna() & (loans['returned'] == False)).sum()

print("\nMissing return_date & book IS back (returned == True):", missing_and_returned)
print("Missing return_date & book IS NOT back (returned == False):", missing_and_not_returned)

# 30 loans where return_date is missing and returned == False are "missing for a reason" 
# becaus the book hasn't been returned yet, so there cannot be a return date
# 6 loans where return_date is missing and returned == True are "missing by accident" 
# because the book was returned, but the date was forgotten/lost in the system

# Create a column has_return_date with notna() and count it
loans['has_return_date'] = loans['return_date'].notna()
print(loans['has_return_date'].value_counts())

Task 3

In [ ]:
members_raw = pd.read_csv('data/library_members.csv')
members = members_raw.copy()

print(members.shape)
print(members.info())

print("\nbirth_year:")
# print(members['birth_year'].info())
print(members['birth_year'].describe())

# Count strange values in birth_year
print("\nCount of 1900:", (members['birth_year'] == 1900).sum())
print("Count of 2030:", (members['birth_year'] == 2030).sum())

# Check duplicates in whole row and in member_id
print("\nDuplicate rows:", members.duplicated().sum())
print("Duplicate member_id:", members.duplicated(subset=['member_id']).sum())

# List of problems:
# 1. Missing values: branch has 12 missing values, birth_year has 18
# 2. Impossible values of birth_year: 8 members have 1900 and 3 members have 2030
# 3. Duplicates: 4 exact duplicate rows and 4 duplicate member_id


Task 4

In [ ]:
# Turn those values into missing values
members['birth_year'] = members['birth_year'].where(members['birth_year'].between(1920, 2015))

# Count how many birth years are missing now
print("Missing birth_year:", members['birth_year'].isna().sum())

# Run describe() again and compare with before
print("\nDescribe birth_year after cleaning:")
print(members['birth_year'].describe())

# Missing values: 18 -> 29 
# min: 1900  -> 1957, max 2030 -> 2010

Task 5

In [ ]:
# (a) Without saving the result, see how many rows are left with dropna()
print("dropna():", members.dropna().shape)
print("dropna branch:", members.dropna(subset=['branch']).shape)
print("dropna birth_year:", members.dropna(subset=['birth_year']).shape)

# (b) Fill the missing branch with 'Unknown' and count the branches
members['branch'] = members['branch'].fillna('Unknown')
print(f"\nBranches: {members['branch'].value_counts()}")

# (c)Try filling the missing birth_year with the median
median = members['birth_year'].median()
birth_filled = members['birth_year'].fillna(median)

print("\nMedian:", median)
print("Before: mean =", round(members['birth_year'].mean(), 2), "std =", round(members['birth_year'].std(), 2))
print("After:  mean =", round(birth_filled.mean(), 2), "std =", round(birth_filled.std(), 2))

# Spread decreased from 12.06 to 10.81 because we added 29 median values
# Better to leave birth_year as NaN so we don't distort the real age distribution

Task 6

In [ ]:
print("loans shape:", loans.shape)
print("members shape:", members.shape)

merged_bad = loans.merge(members, on='member_id', how='left')
print("\nMerged shape:", merged_bad.shape)
# We get 616 rows instead of 600 because 4 members appear twice in members so their loans duplicated

print("late_fee before:", loans['late_fee'].sum())
print("late_fee after:", merged_bad['late_fee'].sum())

# loans.merge(members, on='member_id', how='left', validate='many_to_one')
# MergeError: Merge keys are not unique in right dataset; not a many-to-one merge
